# 05. Базовая визуализация аналитических данных

## Тема

**Загрузка и интеграция данных из различных форматов. Инструменты для сбора данных. Основы Python для обработки данных**

В предыдущем ноутбуке мы научились считать показатели:

- выручку;
- прибыль;
- группировки по категориям;
- группировки по регионам;
- сводные таблицы;
- корреляции.

Теперь научимся визуально проверять результаты анализа.

В этом ноутбуке оставляем только графики, которые реально полезны начинающему аналитику:

1. динамика продаж;
2. продажи по категориям;
3. топ регионов;
4. распределение чеков;
5. boxplot по категориям;
6. scatter plot.

## 1. Цель ноутбука

После выполнения ноутбука вы должны уметь:

1. Загружать подготовленный датасет `sales_prepared.csv`.
2. Готовить данные для графика через `groupby`.
3. Строить линейный график динамики продаж.
4. Строить столбчатую диаграмму продаж по категориям.
5. Строить горизонтальную диаграмму топ-регионов.
6. Строить гистограмму распределения чеков.
7. Строить boxplot для сравнения категорий.
8. Строить scatter plot для проверки связи показателей.
9. Подписывать заголовки, оси и поворачивать подписи.
10. Сохранять графики в PNG-файлы.

Главная идея:

> график нужен не «для красоты», а для ответа на аналитический вопрос.

## 2. Как выбирать тип графика

| Аналитический вопрос | Подходящий график |
|---|---|
| Как меняется показатель во времени? | Линейный график |
| Какая категория больше/меньше? | Столбчатая диаграмма |
| Какие регионы входят в топ? | Горизонтальная столбчатая диаграмма |
| Как распределены чеки? | Гистограмма |
| Есть ли выбросы внутри категорий? | Boxplot |
| Есть ли связь между двумя числовыми показателями? | Scatter plot |

В этом ноутбуке все графики будут связаны с единым бизнес-кейсом **«РегионМаркет»**.

## 3. Импорт библиотек

Используем:

- `pandas` — подготовка таблиц для графиков;
- `matplotlib.pyplot` — построение графиков;
- `Path` — работа с путями.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("pandas:", pd.__version__)

## 4. Поиск подготовленного файла

Нам нужен файл:

```text
data/prepared/sales_prepared.csv
```

Если его нет, сначала выполните ноутбук:

```text
03_data_integration.ipynb
```

In [ ]:
def find_prepared_file() -> Path:
    """Найти файл sales_prepared.csv в типовых местах."""
    current_dir = Path.cwd()

    candidates = [
        current_dir / "data" / "prepared" / "sales_prepared.csv",
        current_dir.parent / "data" / "prepared" / "sales_prepared.csv",
        current_dir.parent.parent / "data" / "prepared" / "sales_prepared.csv",
        current_dir / "sales_prepared.csv",
        current_dir.parent / "sales_prepared.csv",
        Path("/mnt/data/data/prepared/sales_prepared.csv"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return current_dir / "data" / "prepared" / "sales_prepared.csv"


prepared_path = find_prepared_file()

print("Файл:")
print(prepared_path)

if not prepared_path.exists():
    raise FileNotFoundError(
        "Файл sales_prepared.csv не найден. "
        "Сначала выполните 03_data_integration.ipynb."
    )

## 5. Загрузка данных

In [ ]:
df = pd.read_csv(prepared_path)

print("Размер таблицы:", df.shape)

df.head()

## 6. Первичная подготовка данных для визуализации

После загрузки CSV даты снова могут быть текстом.  
Преобразуем дату заказа и убедимся, что есть показатель `revenue`.

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

if "revenue" not in df.columns:
    df["revenue"] = df["quantity"] * df["unit_price"] * (1 - df["discount_percent"] / 100)

print("Тип order_date:", df["order_date"].dtype)
print("Есть revenue:", "revenue" in df.columns)

df[["order_date", "quantity", "unit_price", "discount_percent", "revenue"]].head()

## 7. Уберем строки, которые не подходят для графиков

В учебных данных специально есть ошибки.  
Для графиков отберем строки, где:

- дата распознана;
- `revenue` не пустая;
- `revenue` больше 0.

In [ ]:
viz_df = df[
    df["order_date"].notna() &
    df["revenue"].notna() &
    (df["revenue"] > 0)
].copy()

print("Строк в исходной таблице:", df.shape[0])
print("Строк для визуализации:", viz_df.shape[0])

# Часть 1. Динамика продаж

## 8. Аналитический вопрос

> Как менялась выручка во времени?

Для ответа используем линейный график.

Сначала подготовим данные: сгруппируем продажи по датам.

In [ ]:
daily_revenue = (
    viz_df
    .groupby("order_date", as_index=False)
    .agg(total_revenue=("revenue", "sum"))
    .sort_values("order_date")
)

daily_revenue.head()

## 9. Линейный график динамики продаж

Линейный график хорошо подходит для временных рядов.

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(daily_revenue["order_date"], daily_revenue["total_revenue"], marker="o")

plt.title("Динамика выручки по датам")
plt.xlabel("Дата заказа")
plt.ylabel("Выручка")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()

plt.show()

### Как читать график

На этом графике можно увидеть:

- дни с высокой выручкой;
- дни с низкой выручкой;
- резкие скачки;
- возможные аномалии.

Важно: если в один день выручка резко выше остальных, нужно проверить, не связано ли это с крупным B2B-заказом или ошибкой данных.

# Часть 2. Продажи по категориям

## 10. Аналитический вопрос

> Какие категории товаров дают основную выручку?

Для ответа используем столбчатую диаграмму.

In [ ]:
category_revenue = (
    viz_df
    .groupby("category", dropna=False, as_index=False)
    .agg(total_revenue=("revenue", "sum"))
    .sort_values("total_revenue", ascending=False)
)

category_revenue

## 11. Столбчатая диаграмма по категориям

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(category_revenue["category"].astype(str), category_revenue["total_revenue"])

plt.title("Выручка по категориям")
plt.xlabel("Категория")
plt.ylabel("Выручка")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

### Как читать график

Столбчатая диаграмма помогает быстро сравнить категории.

Вопросы для аналитика:

- какая категория лидирует;
- насколько сильно она опережает остальные;
- есть ли категории с очень низкой выручкой;
- стоит ли проверять прибыльность, а не только выручку.

# Часть 3. Топ регионов

## 12. Аналитический вопрос

> Какие регионы дают максимальную выручку?

Для рейтингов часто удобнее использовать горизонтальную столбчатую диаграмму.

In [ ]:
top_regions = (
    viz_df
    .groupby("region_name", dropna=False, as_index=False)
    .agg(total_revenue=("revenue", "sum"))
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

top_regions

## 13. Горизонтальная диаграмма топ-регионов

In [ ]:
top_regions_for_plot = top_regions.sort_values("total_revenue")

plt.figure(figsize=(10, 6))

plt.barh(top_regions_for_plot["region_name"].astype(str), top_regions_for_plot["total_revenue"])

plt.title("Топ регионов по выручке")
plt.xlabel("Выручка")
plt.ylabel("Регион")
plt.tight_layout()

plt.show()

### Как читать график

Горизонтальная диаграмма удобна, когда подписи длинные.

Вопросы для аналитика:

- какой регион лидер;
- насколько лидер отрывается от остальных;
- есть ли регионы с низким вкладом;
- совпадает ли топ регионов с планами продаж.

# Часть 4. Распределение чеков

## 14. Аналитический вопрос

> Как распределены суммы заказов?

Для ответа используем гистограмму.

В этом ноутбуке под чеком будем понимать `revenue` по одной строке продажи.

In [ ]:
order_revenue = viz_df["revenue"].dropna()

print("Количество значений:", order_revenue.shape[0])
print("Минимум:", order_revenue.min())
print("Максимум:", order_revenue.max())
print("Среднее:", order_revenue.mean())
print("Медиана:", order_revenue.median())

## 15. Гистограмма распределения чеков

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(order_revenue, bins=15)

plt.title("Распределение суммы заказа")
plt.xlabel("Сумма заказа")
plt.ylabel("Количество заказов")
plt.tight_layout()

plt.show()

### Как читать гистограмму

Гистограмма помогает понять:

- каких заказов больше: маленьких, средних или крупных;
- есть ли длинный хвост крупных заказов;
- отличается ли среднее от медианы;
- есть ли подозрительные значения.

Если есть один очень крупный заказ, он может сильно влиять на среднее.

# Часть 5. Boxplot по категориям

## 16. Аналитический вопрос

> Как различаются суммы заказов внутри категорий и есть ли выбросы?

Для этого используем boxplot.

Boxplot показывает:

- медиану;
- основной диапазон значений;
- потенциальные выбросы.

In [ ]:
boxplot_data = []

boxplot_labels = []

for category, group in viz_df.groupby("category", dropna=False):
    values = group["revenue"].dropna()
    if not values.empty:
        boxplot_data.append(values)
        boxplot_labels.append(str(category))

print("Категории для boxplot:")
print(boxplot_labels)

## 17. Boxplot суммы заказа по категориям

In [ ]:
plt.figure(figsize=(10, 5))

plt.boxplot(boxplot_data, labels=boxplot_labels)

plt.title("Распределение суммы заказа по категориям")
plt.xlabel("Категория")
plt.ylabel("Сумма заказа")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

### Как читать boxplot

Boxplot помогает увидеть:

- в какой категории медианный чек выше;
- где разброс значений больше;
- где есть выбросы;
- какие категории требуют дополнительной проверки.

Важно: выброс не всегда ошибка. Это может быть реальный крупный заказ.

# Часть 6. Scatter plot

## 18. Аналитический вопрос

> Есть ли связь между ценой товара и количеством проданных единиц?

Для ответа используем scatter plot.

На scatter plot каждая точка — одна продажа.

In [ ]:
scatter_df = viz_df[
    viz_df["unit_price"].notna() &
    viz_df["quantity"].notna()
].copy()

print("Строк для scatter plot:", scatter_df.shape[0])

scatter_df[["unit_price", "quantity", "revenue"]].head()

## 19. Scatter plot: цена и количество

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(scatter_df["unit_price"], scatter_df["quantity"])

plt.title("Связь цены товара и количества")
plt.xlabel("Цена за единицу")
plt.ylabel("Количество")
plt.grid(True)
plt.tight_layout()

plt.show()

### Как читать scatter plot

Scatter plot помогает увидеть:

- есть ли связь между двумя числовыми показателями;
- есть ли выбросы;
- есть ли группы точек;
- есть ли подозрительные значения.

Возможная гипотеза:

> дорогие товары покупают меньшим количеством, а недорогие товары — большим.

Но по одному графику нельзя делать окончательный вывод. Его нужно проверять расчетами и бизнес-контекстом.

## 20. Scatter plot: выручка и прибыль

Проверим связь между выручкой и валовой прибылью.

In [ ]:
profit_scatter_df = viz_df[
    viz_df["revenue"].notna() &
    viz_df["gross_profit"].notna()
].copy()

plt.figure(figsize=(8, 5))

plt.scatter(profit_scatter_df["revenue"], profit_scatter_df["gross_profit"])

plt.title("Связь выручки и валовой прибыли")
plt.xlabel("Выручка")
plt.ylabel("Валовая прибыль")
plt.grid(True)
plt.tight_layout()

plt.show()

### Как читать график

Если точки идут вверх слева направо, значит большая выручка часто связана с большей прибылью.

Но если есть точки с высокой выручкой и низкой прибылью, стоит проверить:

- большие скидки;
- высокую закупочную цену;
- возвраты;
- ошибки в данных.

# Часть 7. Сохранение графиков

## 21. Папка для графиков

Создадим папку:

```text
data/output/figures
```

In [ ]:
figures_dir = Path("data/output/figures")
figures_dir.mkdir(parents=True, exist_ok=True)

print("Папка для графиков:")
print(figures_dir)

## 22. Сохранение графика динамики продаж

Чтобы сохранить график, используем:

```python
plt.savefig(...)
```

Важно: `savefig()` нужно вызывать до `plt.show()`.

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(daily_revenue["order_date"], daily_revenue["total_revenue"], marker="o")

plt.title("Динамика выручки по датам")
plt.xlabel("Дата заказа")
plt.ylabel("Выручка")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()

dynamic_path = figures_dir / "daily_revenue.png"
plt.savefig(dynamic_path, dpi=150)

plt.show()

print("Файл сохранен:", dynamic_path)

## 23. Сохранение графика категорий

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(category_revenue["category"].astype(str), category_revenue["total_revenue"])

plt.title("Выручка по категориям")
plt.xlabel("Категория")
plt.ylabel("Выручка")
plt.xticks(rotation=45)
plt.tight_layout()

category_path = figures_dir / "category_revenue.png"
plt.savefig(category_path, dpi=150)

plt.show()

print("Файл сохранен:", category_path)

# Часть 8. Шпаргалка по графикам

| График | Команда | Когда использовать |
|---|---|---|
| Линейный график | `plt.plot()` | Динамика во времени |
| Столбчатая диаграмма | `plt.bar()` | Сравнение категорий |
| Горизонтальная диаграмма | `plt.barh()` | Рейтинги и длинные подписи |
| Гистограмма | `plt.hist()` | Распределение числового показателя |
| Boxplot | `plt.boxplot()` | Разброс и выбросы по группам |
| Scatter plot | `plt.scatter()` | Связь двух числовых показателей |

# Часть 9. Типовые ошибки

## Ошибка 1. `plt.show` без скобок

Неправильно:

```python
plt.show
```

Правильно:

```python
plt.show()
```

---

## Ошибка 2. График пустой

Проверьте:

```python
df.shape
df.isna().sum()
```

Возможно, после фильтрации не осталось строк.

---

## Ошибка 3. Даты на оси X отображаются плохо

Используйте:

```python
plt.xticks(rotation=45)
plt.tight_layout()
```

---

## Ошибка 4. Слишком много категорий

Ограничьте данные топом:

```python
.head(10)
```

---

## Ошибка 5. Boxplot не строится

Проверьте, что в каждой группе есть числовые значения:

```python
df["revenue"].dtype
df["revenue"].isna().sum()
```

# Часть 10. Мини-задания

Выполните задания самостоятельно.

## Задание 1

Постройте столбчатую диаграмму выручки по каналам продаж.

In [ ]:
# Ваш код здесь

## Задание 2

Постройте горизонтальную диаграмму топ-5 товаров по выручке.

In [ ]:
# Ваш код здесь

## Задание 3

Постройте гистограмму `gross_profit`.

Перед построением удалите пропуски.

In [ ]:
# Ваш код здесь

## Задание 4

Постройте boxplot `revenue` по `channel`.

In [ ]:
# Ваш код здесь

## Задание 5

Постройте scatter plot:

```text
discount_percent × revenue
```

Подумайте, есть ли визуальная связь между скидкой и выручкой.

In [ ]:
# Ваш код здесь

## Задание 6

Сохраните любой построенный график в папку:

```text
data/output/figures
```

In [ ]:
# Ваш код здесь

# Часть 11. Контрольные вопросы

Ответьте своими словами:

1. Для чего нужен линейный график?
2. Когда лучше использовать столбчатую диаграмму?
3. Почему для топ-регионов удобна горизонтальная диаграмма?
4. Что показывает гистограмма?
5. Что показывает boxplot?
6. Что показывает scatter plot?
7. Почему график не заменяет расчет показателей?
8. Почему выброс на графике не всегда ошибка?
9. Зачем подписывать оси графика?
10. Зачем сохранять графики в файлы?

# 24. Итог ноутбука

В этом ноутбуке мы научились строить только те графики, которые полезны для первичного анализа:

- динамика продаж;
- продажи по категориям;
- топ регионов;
- распределение чеков;
- boxplot по категориям;
- scatter plot.

Главная идея:

> визуализация должна отвечать на аналитический вопрос, а не просто украшать отчет.

Следующий шаг:

```text
06_python_to_r_bridge.ipynb
```

Там мы покажем, как результаты Python-обработки можно передать в R.